# Decorador AR para interiores

Decora fotografías reales con cuadros, alfombras, muebles 3D y objetos sobre encimeras. El cuaderno muestra cómo se obtienen las superficies, cómo se proyectan las piezas y qué errores medimos. Ejecuta las celdas en orden con el kernel de `.venv`.

La primera ejecución prepara las vistas de [ZInD](https://github.com/zillow/zind/tree/main/sample_tour/000). El ZIP de ADE20K se usa si está disponible. Los PNG y las mediciones se guardan en una carpeta nueva.

In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys
from importlib.metadata import version

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/decorador_ar').is_dir())
sys.path.insert(0, str(root / 'src'))

import cv2 as cv
import matplotlib.pyplot as plt
from decorador_ar.photo import Segmenter
from decorador_ar.depth import evaluate_files
from decorador_ar.image_io import write_image
from decorador_ar.editor_web import display_editor, start_editor
from decorador_ar import course_experiments as experiments
from decorador_ar.photo_furniture import MODELS
from decorador_ar import notebook_workflow as workflow

SCENE = 'bonus_02'
output = workflow.new_run(root)
segmenter = Segmenter()
print(f'Resultados en {output}')
print('Python', sys.version.split()[0], '· OpenCV', cv.__version__,
      '· NumPy', version('numpy'), '· Transformers', version('transformers'))
print('SegFormer:', segmenter.revision)

## 1. Foto real, pared y suelo

Empezamos con seis vistas de una vivienda y añadimos otras seis en el bloque 3D. SegFormer identifica pared y suelo. Las aristas anotadas por ZInD dan la referencia geométrica para rectificar ambos planos, como en la práctica 1.

In [ ]:
scenes = workflow.load_scenes(root, segmenter)
print('Vistas disponibles:', ', '.join(scenes))
scene = scenes[SCENE]
workflow.show_original(scene, SCENE)
plt.show()

## 2. Rectificar y diseñar la habitación · BT2a, BT2b

La línea amarilla debe seguir el rodapié en la foto y quedar horizontal tras la rectificación. La transformación permite definir el tamaño y giro de la pieza en su plano. La máscara comprueba si la pieza queda sobre su superficie. Las seis composiciones usan puntos elegidos para cada foto.

In [ ]:
workflow.show_rectification(scene)
plt.show()
workflow.show_gallery(scenes)
plt.show()

## 3. Evaluar la segmentación · BT3d, BT3g, BT5d, BT5f, BT5h

Con el ZIP de ADE20K se calcula IoU de pared y suelo en nueve imágenes. En tres fotos fijas se comparan argmax, apertura y cierre, y un umbral de probabilidad de 0.55. La implementación NumPy se contrasta píxel a píxel con OpenCV. Con `COMPARE_CNN = True` se añade un segmentador convolucional [ConvNeXt-Tiny + UperNet](https://huggingface.co/openmmlab/upernet-convnext-tiny) sobre las mismas imágenes y anotaciones. La primera descarga de la CNN ocupa unos 241 MB.

In [ ]:
COMPARE_CNN = True
archive = root / 'data/raw/ADEChallengeData2016.zip'
if archive.is_file():
    scores = workflow.evaluate_masks(archive, output / 'ade20k', segmenter)
    print('IoU media por superficie:', scores)
    print(output / 'ade20k/resumen.csv')
    experiments.mask_comparison(archive, segmenter, compare_cnn=COMPARE_CNN)
    plt.show()
else:
    print('Evaluación ADE20K disponible al guardar el ZIP en data/raw/.')

## 4. Colocar muebles 3D en fotos

Proyectamos una mesa baja y un banco, con patas y tablero, usando la cámara y el plano del suelo de ZInD. La huella debe quedar sobre suelo detectado. En las seis vistas nuevas incluimos dos pasillos para comprobar por qué el encaje de la base no basta para decidir si un mueble estorba. Las doce vistas vienen de una sola vivienda.

Para probar otra posición, cambia el centro en píxeles, el modelo, la escala o el giro. El render ordena las superficies del propio mueble por profundidad. La ocultación detrás de objetos reales se incorporará al medir la profundidad de la habitación.

In [ ]:
stock = workflow.load_furniture_stock(root, segmenter)
workflow.show_furniture_stock(stock)

FURNITURE_SCENE = 'bonus_02'
MODEL, SCALE, YAW = 'mesa_baja', .73, 0
ANCHOR = (416, 576)  # Centro de apoyo (x, y). Pon None para usar el punto de control.
print('Modelos:', ', '.join(MODELS))
furniture_scene = (scenes | stock)[FURNITURE_SCENE]
base = workflow.decorate_2d(FURNITURE_SCENE, furniture_scene) if FURNITURE_SCENE in scenes else None
try:
    preview = workflow.furniture_preview(furniture_scene, ANCHOR or furniture_scene.floor_anchor,
                                         model=MODEL, scale=SCALE, yaw=YAW, image=base)
except ValueError as error:
    print('Colocación rechazada:', error)
else:
    workflow.show_furniture_detail(furniture_scene, preview)
    plt.show()
    write_image(output / 'mueble-3d.png', preview.image)

## 5. Registrar un cuadro entre dos vistas · BT2e, BT4a, BT4b, BT4d, BT7a, BT7c

ORB localiza puntos y compara sus descriptores. Se ajusta una homografía con todas las coincidencias y otra con RANSAC, que descarta las que superan 3 píxeles de error. En dos vistas reales del mismo panorama conocemos el giro usado al generarlas. Con él obtenemos una referencia cuantitativa para puntos que quedan dentro de ambas vistas y para las cuatro esquinas del cuadro.

Este control evalúa una cámara que gira desde el mismo centro. La traslación entre fotos y los cambios de punto de vista con paralaje quedan por validar.

In [ ]:
experiments.show_registration(scenes)
plt.show()

## 6. Apoyos reales y luz local · BT3a, BT3d, BT6a, BT6f

SegFormer señala posibles muebles y Depth Anything estima su distancia. Se usa el suelo para ajustar la escala. La profundidad permite buscar regiones elevadas que sean aproximadamente horizontales. La morfología agrupa sus píxeles y se busca una zona rectangular interior sobre la que quepa la pieza. El contorno cian enseña qué parte de la encimera eligió el método.

Las tres imágenes separan la detección, la proyección del plato y el ajuste de iluminación. La última aplica brillo y color del entorno y una sombra corta donde la base toca la encimera. Es una aproximación visual sin estimar la dirección de cada fuente de luz.

In [ ]:
support_scenes = workflow.load_support_scenes(root, segmenter)
extra = workflow.load_extra_scenes(root, segmenter)
workflow.show_photo_stock(extra)
plt.show()

try:
    experiments.show_support_comparison(support_scenes['cocina_01'])
except (OSError, ValueError) as error:
    print('Demostración de profundidad no disponible:', error)
else:
    plt.show()

## 7. Evaluar profundidad cuando haya mapas medidos · BT6f

Esta celda compara mapas registrados y expresados en metros. El experimento de la encimera muestra que la predicción sirve para proponer una zona. El error métrico y la ocultación por objetos de la foto requieren una referencia RGB-D.

In [ ]:
DEPTH_REFERENCE = None  # Mapa .npy de referencia.
DEPTH_PREDICTION = None  # Predicción alineada.
if DEPTH_REFERENCE is not None and DEPTH_PREDICTION is not None:
    print(evaluate_files(DEPTH_REFERENCE, DEPTH_PREDICTION, output / 'profundidad.json', .1, 10))

## 8. Componer una habitación · BT1c

Elige **Habitación**, **Superficie** y **Pieza**. La pieza aparece translúcida en una posición inicial. Mueve el cursor sobre la foto para desplazarla y haz clic para dejarla puesta. La pieza puede sobresalir del encuadre si la parte visible encaja en su superficie. Los controles de escala, giro y luz están en **Ajustes**. También puedes enfocar la foto, mover la pieza con las flechas y fijarla con **Enter**.

Para probar una encimera, elige **cocina 01 → En altura → Detectar apoyos → Plato**. El contorno cian marca el apoyo. Los muebles virtuales con tablero ofrecen su propio apoyo y **Ajustes → Marcar tablero** permite señalar cuatro esquinas de una superficie real.

Si PyCharm oculta la vista integrada, abre el enlace del navegador o copia la dirección local que imprime la celda. Deja el kernel activo mientras editas. **Guardar PNG** guarda las piezas colocadas.

In [ ]:
if os.getenv('DECORADOR_AR_NOTEBOOK_OUTPUT'):
    print('Para editar, abre este notebook en PyCharm y ejecuta la celda con el kernel activo.')
else:
    if 'editor_server' in globals():
        editor_server.shutdown()
        editor_server.server_close()
    editor_server = start_editor(scenes | stock | support_scenes | extra, SCENE, output)
    print(editor_server.url)
    display_editor(editor_server)

In [ ]:
print(f'Las composiciones exportadas se guardan en {output}')

## Cobertura del enunciado

Los códigos y pesos proceden de la **sección 6 de Enunciados de Prácticas 2026/2027**. Se indica la evidencia que puedes volver a ejecutar y la ampliación que queda por validar.

| Bloque | Tópicos con evidencia en este cuaderno | Qué falta o ampliaría la evidencia |
| --- | --- | --- |
| BT1 · 15 % | **a, b, c.** Versiones, parámetros, figuras, mediciones y editor interactivo. | Reservar imágenes de otras viviendas y fijar su manifiesto. |
| BT2 · 20 % | **a, b, e, f.** Homografías, pared y suelo rectificados, registro con giro conocido y proyección de la decoración. | **c, d.** Calibrar la cámara propia, validar pose y reproyección con esquinas independientes y repetir el registro con traslación. |
| BT3 · 20 % | **a, d, g.** Sombra suavizada, morfología y regiones conexas para apoyos, y apertura/cierre propios comparados con OpenCV sobre ADE20K. | Medir cuánto cambia la detección de encimeras si se omite la morfología. |
| BT4 · 15 % | **a, b, d.** Puntos ORB, descriptores binarios y traslado del contorno de un cuadro con correspondencias entre vistas. | Comparar texturas difíciles y habitaciones de otras viviendas. |
| BT5 · 15 % | **d, f, h.** IoU, comparación de un segmentador CNN con SegFormer, mapa de probabilidad, argmax y efecto de un umbral. | Ampliar la comparación CNN a una muestra reservada y medir el tiempo de inferencia. |
| BT6 · 10 % | **a, f.** Segmentación con SegFormer y profundidad monocular aplicada a elegir un apoyo real. | Medir error de profundidad y de altura frente a RGB-D y validar apoyos en otras viviendas. |
| BT7 · 10 % | **a, c.** RANSAC y ajuste sin rechazo robusto sobre las mismas correspondencias. | Repetir el experimento con otras vistas y estudiar el efecto del umbral. |

**Lectura provisional.** Los candidatos con evidencia suman 105 %. El enunciado pide 100 % en continua o 120 % en global. La suma es una propuesta de cobertura pendiente del criterio del profesor. Las prioridades son comprobar la geometría con fotos propias y traslación, validar apoyos en otras viviendas y ampliar la comparación de máscaras. Para alcanzar 120 % en global, tres ampliaciones candidatas son calibrar cámara (BT2c), validar pose (BT2d) y medir la sensibilidad al umbral de RANSAC (BT7d).